In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction import FeatureHasher
from sklearn.preprocessing import PolynomialFeatures

sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv('../data/Train.csv')
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1000, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,C,Flight,2,5,306,6,high,M,45,3838,1
1,2,F,Ship,7,2,114,3,high,M,35,2710,1
2,3,B,Ship,7,2,215,7,medium,F,44,4152,0
3,4,A,Road,5,3,126,5,medium,M,54,2245,0
4,5,F,Ship,3,5,113,3,medium,M,43,1806,1


In [3]:
print("Column\n", df.columns) #column
print("Missing Value\n", df.isnull().sum())

Column
 Index(['ID', 'Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls',
       'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases',
       'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms',
       'Reached.on.Time_Y.N'],
      dtype='object')
Missing Value
 ID                     0
Warehouse_block        0
Mode_of_Shipment       0
Customer_care_calls    0
Customer_rating        0
Cost_of_the_Product    0
Prior_purchases        0
Product_importance     0
Gender                 0
Discount_offered       0
Weight_in_gms          0
Reached.on.Time_Y.N    0
dtype: int64


In [4]:
mode_list = df['Mode_of_Shipment'].astype(str).apply(lambda x: [x])

hasher = FeatureHasher(n_features=4, input_type='string')

hashed_features = hasher.transform(mode_list)

hashed_df = pd.DataFrame(
    hashed_features.toarray(),
    columns=[f"Mode_hash_{i}" for i in range(4)]
)

print("Hashed Features Shape:", hashed_df.shape)
hashed_df.head()

Hashed Features Shape: (1000, 4)


,Mode_hash_0,Mode_hash_1,Mode_hash_2,Mode_hash_3
0,0.0,0.0,-1.0,0.0
1,-1.0,0.0,0.0,0.0
2,-1.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0
4,-1.0,0.0,0.0,0.0


In [5]:
cat_cols = ['Mode_of_Shipment', 'Warehouse_block']
combined = df[cat_cols].astype(str).agg(' '.join, axis=1)
combined_tokens = combined.apply(lambda x: x.split())

hasher_multi = FeatureHasher(n_features=6, input_type='string')

hashed_multi = hasher_multi.transform(combined_tokens)
hashed_multi_df = pd.DataFrame(
    hashed_multi.toarray(),
    columns=[f"Hash_{i}" for i in range(6)]
)
print("Hashed Multi Shape:", hashed_multi_df.shape)
hashed_multi_df.head()

Hashed Multi Shape: (1000, 6)


,Hash_0,Hash_1,Hash_2,Hash_3,Hash_4,Hash_5
0,0.0,0.0,-1.0,-1.0,0.0,0.0
1,0.0,0.0,-2.0,0.0,0.0,0.0
2,0.0,0.0,-2.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,1.0,0.0
4,0.0,0.0,-2.0,0.0,0.0,0.0


In [6]:
df['Cost_per_Gram'] = df['Cost_of_the_Product'] / (df['Weight_in_gms'] + 1)

df[['Cost_of_the_Product','Weight_in_gms','Cost_per_Gram']].head()

,Cost_of_the_Product,Weight_in_gms,Cost_per_Gram
0,306,3838,0.079708
1,114,2710,0.042051
2,215,4152,0.051770
3,126,2245,0.056100
4,113,1806,0.062535


In [7]:
df['Discount_Ratio'] = df['Discount_offered'] / (df['Cost_of_the_Product'] + 1)

df[['Discount_offered','Cost_of_the_Product','Discount_Ratio']].head()

,Discount_offered,Cost_of_the_Product,Discount_Ratio
0,45,306,0.146580
1,35,114,0.304348
2,44,215,0.203704
3,54,126,0.425197
4,43,113,0.377193


In [8]:
df['Customer_Engagement'] = df['Customer_care_calls'] * df['Customer_rating']

df[['Customer_care_calls','Customer_rating','Customer_Engagement']].head()

,Customer_care_calls,Customer_rating,Customer_Engagement
0,2,5,10
1,7,2,14
2,7,2,14
3,5,3,15
4,3,5,15


In [9]:
poly = PolynomialFeatures(degree=2, interaction_only=True)

cols = ['Cost_of_the_Product','Discount_offered']

poly_features = poly.fit_transform(df[cols])

poly_feature_names = poly.get_feature_names_out(cols)

poly_df = pd.DataFrame(poly_features, columns=poly_feature_names)

poly_df.head()

,1,Cost_of_the_Product,Discount_offered,Cost_of_the_Product Discount_offered
0,1.0,306.0,45.0,13770.0
1,1.0,114.0,35.0,3990.0
2,1.0,215.0,44.0,9460.0
3,1.0,126.0,54.0,6804.0
4,1.0,113.0,43.0,4859.0


In [10]:
df_final = pd.concat([df, hashed_df, hashed_multi_df, poly_df], axis=1)

print("Final Dataset Shape:", df_final.shape)
df_final.head()

Final Dataset Shape: (1000, 29)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,...,Hash_0,Hash_1,Hash_2,Hash_3,Hash_4,Hash_5,1,Cost_of_the_Product,Discount_offered,Cost_of_the_Product Discount_offered
0,1,C,Flight,2,5,306,6,high,M,45,...,0.0,0.0,-1.0,-1.0,0.0,0.0,1.0,306.0,45.0,13770.0
1,2,F,Ship,7,2,114,3,high,M,35,...,0.0,0.0,-2.0,0.0,0.0,0.0,1.0,114.0,35.0,3990.0
2,3,B,Ship,7,2,215,7,medium,F,44,...,0.0,0.0,-2.0,0.0,0.0,0.0,1.0,215.0,44.0,9460.0
3,4,A,Road,5,3,126,5,medium,M,54,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,126.0,54.0,6804.0
4,5,F,Ship,3,5,113,3,medium,M,43,...,0.0,0.0,-2.0,0.0,0.0,0.0,1.0,113.0,43.0,4859.0
